# 🏥 ClinicQueue — Fine-tune flan-t5-base for Triage

**Estimated time on free T4 GPU: ~15–20 minutes**

### Steps:
1. Install dependencies
2. Upload your `triage_dataset.json` from your PC
3. Run training (all cells below)
4. Download the trained model ZIP
5. Put it in `server/ai-model/triage_model/` on your PC

---
**Before running:** Go to `Runtime → Change runtime type → T4 GPU → Save`

## Step 1 — Install dependencies

In [ ]:
!pip install -q transformers datasets accelerate sentencepiece
print('✅ Dependencies installed')

## Step 2 — Upload your dataset
Run this cell, then click **Choose Files** and upload `triage_dataset.json` from your PC.

Your file is at: `d:\ClinicQueue\server\ai-model\triage_dataset.json`

In [ ]:
from google.colab import files
uploaded = files.upload()
print('✅ File uploaded:', list(uploaded.keys()))

## Step 3 — Verify GPU is available

In [ ]:
import torch
print(f'GPU available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name      : {torch.cuda.get_device_name(0)}')
    print(f'GPU memory    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU — go to Runtime → Change runtime type → T4 GPU')

## Step 4 — Load & Deduplicate Dataset

In [ ]:
import json, random
from collections import Counter

SEED = 42
random.seed(SEED)

with open('triage_dataset.json', encoding='utf-8') as f:
    raw_data = json.load(f)

print(f'Loaded       : {len(raw_data):,} examples')

# Deduplicate
seen = set()
data = []
for entry in raw_data:
    key = entry['input'].strip().lower()
    if key not in seen:
        seen.add(key)
        data.append(entry)

print(f'After dedup  : {len(data):,} examples  (removed {len(raw_data)-len(data)} duplicates)')

# Class distribution
scores = [json.loads(e['output'])['severityScore'] for e in data]
print('\nClass distribution:')
for lvl, cnt in sorted(Counter(scores).items()):
    print(f'  Level {lvl}: {cnt:4d}  {"#" * (cnt // 30)}')

## Step 5 — Train / Validation Split

In [ ]:
random.shuffle(data)
split = int(len(data) * 0.85)
train_data = data[:split]
val_data   = data[split:]
print(f'Train : {len(train_data):,} examples')
print(f'Val   : {len(val_data):,} examples')

## Step 6 — Load flan-t5-base & Tokeniser

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

MODEL_NAME = 'google/flan-t5-base'

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model     = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

params = sum(p.numel() for p in model.parameters())
print(f'Model loaded  : {MODEL_NAME}')
print(f'Parameters    : {params:,}  ({params/1e6:.0f}M)')

## Step 7 — Tokenise

In [ ]:
from datasets import Dataset

MAX_INPUT  = 256
MAX_OUTPUT = 200

def tokenise(batch):
    model_inputs = tokenizer(
        batch['input'],
        max_length=MAX_INPUT,
        truncation=True,
        padding='max_length'
    )
    labels = tokenizer(
        text_target=batch['output'],
        max_length=MAX_OUTPUT,
        truncation=True,
        padding='max_length'
    )
    labels['input_ids'] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in label]
        for label in labels['input_ids']
    ]
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_hf = Dataset.from_list(train_data).map(tokenise, batched=True, batch_size=64, remove_columns=['input','output'])
val_hf   = Dataset.from_list(val_data).map(tokenise,   batched=True, batch_size=64, remove_columns=['input','output'])

print(f'Tokenised train : {len(train_hf):,}')
print(f'Tokenised val   : {len(val_hf):,}')

## Step 8 — Fine-tune (🚀 ~15 mins on T4 GPU)

In [ ]:
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)

OUTPUT_DIR = './triage_model'

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=16,      # bigger batch — GPU has plenty of RAM
    per_device_eval_batch_size=16,
    learning_rate=3e-4,
    weight_decay=0.01,
    warmup_steps=100,
    lr_scheduler_type='cosine',
    predict_with_generate=True,
    generation_max_length=MAX_OUTPUT,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    logging_steps=50,
    seed=42,
    report_to='none',
    fp16=torch.cuda.is_available(),
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_hf,
    eval_dataset=val_hf,
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

steps_per_epoch = len(train_hf) // 16
print(f'Steps / epoch : {steps_per_epoch}')
print(f'Total steps   : {steps_per_epoch * 5}')
print('\n🚀 Starting training...')

trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'\n✅ Model saved to {OUTPUT_DIR}')

## Step 9 — Smoke Test

In [ ]:
model.eval()

tests = [
    ('Severe chest pain, radiating to left arm, sweating, nausea.',         5),
    ('Mild cold, stuffy nose, slight cough, no fever.',                     1),
    ('Sudden numbness on one side of face, slurred speech, confusion.',     5),
    ('Child fever 102, ear pulling, crying, ear infection suspected.',       3),
    ('Deep cut on hand needing stitches, bleeding controlled.',             3),
]

correct = 0
print(f'{"Input":<55} {"Score":>5}  {"Dept":<25}  Result')
print('-' * 105)

for symptoms, expected in tests:
    prompt = f'Triage the following patient symptoms: Patient presents with {symptoms}'
    inputs = tokenizer(prompt, return_tensors='pt', max_length=256, truncation=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200)
    result = tokenizer.decode(out[0], skip_special_tokens=True)
    try:
        parsed = json.loads(result)
        score  = parsed.get('severityScore', '?')
        dept   = parsed.get('recommendedDepartment', '?')
        ok     = '✅ CORRECT' if score == expected else f'❌ (expected {expected})'
        if score == expected: correct += 1
    except:
        score, dept, ok = '?', '?', '❌ JSON ERROR'
    print(f'{symptoms[:53]:<55} {str(score):>5}  {dept:<25}  {ok}')

print(f'\nAccuracy: {correct}/{len(tests)} ({correct/len(tests)*100:.0f}%)')

## Step 10 — Download the trained model
This zips the model and downloads it to your PC.
Then unzip it and place the folder at:
`d:\ClinicQueue\server\ai-model\triage_model\`

In [ ]:
import shutil
shutil.make_archive('triage_model', 'zip', OUTPUT_DIR)
files.download('triage_model.zip')
print('✅ Download started — check your browser downloads folder')